# Moment 3 (part A): long_term_share, and (part B) the SA separation rate

Two outputs from one notebook, per the build plan: `long_term_share` is the fourth `moments.csv`
row, and the separation rate `lambda` is a model *parameter*, not a calibration target -- it's
directly estimated here rather than searched over in Week 4's MSM, the same status
`discount_rate` already has.

## Part A: long_term_share

QLFS's own `Long_term_unempl` classifies every currently-unemployed respondent as long-term
(>= 12 months searching) or short-term, matching `moments.csv`'s definition exactly: the share
of the *currently unemployed* with an in-progress spell of a year or more. Same four quarters,
same person-weighting, same Kish approximate-design-effect SE, and the same
most-recent-quarter-as-headline convention as notebook 01 -- no new methodology here, just the
next pre-built QLFS classification in the same pattern.

## Part B: the SA separation rate

`configs/baseline.yaml` currently carries `separation_rate: 0.0048`, Miyamoto (2011)'s Japanese
monthly rate, explicitly a fallback pending this notebook. The plan's own words: computed "if
the SA rate is not computable by end of week 3." It's computable.

QLFS is a **rotating panel** -- each sampled household is surveyed for several consecutive
quarters before rotating out, and PALMS's own household/person identifiers are only meaningful
*within* a single wave's rotation, not automatically across PALMS's 30-year harmonised span (see
`DECISIONS.md`'s notebook 01/02 entries for the same "don't trust an identifier further than it's
documented to reach" caution). But the raw QLFS quarterly files carry `UQNO` (household) and
`PERSONNO` (within-household), and consecutive quarters really do share sampled households:
linking 2025 Q2 to 2025 Q3 by `UQNO`+`PERSONNO` recovers 43,796 matched individuals out of
65,443 in Q2 -- a genuine ~67 per cent panel retention, not a coincidence at that scale.

**Method**: for each of the three available consecutive quarter-pairs (2025 Q2->Q3, Q3->Q4,
Q4->2026 Q1), take everyone matched across both quarters who was `Employed` in the first quarter,
and check whether they're still `Employed` in the second. The (weighted) share who aren't is that
pair's quarterly separation rate. Pooled across all three pairs (not just averaged -- every
matched employed-then person is one observation in one pooled weighted mean): **9.21% quarterly**.
Rebased to monthly via the same compounding convention already used for `rho_A`
(`(1 - monthly)^3 = (1 - quarterly)`, documented in `DECISIONS.md`'s AR(1) note): **3.17% monthly
(+/- 0.06pp, bootstrapped)** -- roughly 6.6x Miyamoto's Japanese fallback.

**Sanity-checked against StatsSA's own published panel transition figures**, not just trusted
because the computation ran without error: StatsSA reports quarterly employment retention of
91.8% for 2024 Q3->Q4 (implying an 8.2% separation rate that quarter) and 94.0% for 2019 Q3->Q4
(6.0%) -- statssa.gov.za/?p=19090. This notebook's 9.21% for 2025-26 sits close to and slightly
above StatsSA's own 2024 figure, in the same direction as the deteriorating labour market
`discouraged_share` (notebook 01) already showed over the same window. A 6.6x gap from a Japanese
calibration is not a red flag on its own -- Japan is close to the global floor on labour
turnover, so any other country reading several times higher is the expected pattern, not a
surprise.

**Not yet propagated into the config files.** `configs/baseline.yaml`, `trace_demo.yaml` and both
`scarce_vacancies*.yaml` configs were built and validated this session under Miyamoto's 0.0048 --
`trace_demo.yaml`'s specific seed/agent pair was found by scanning under that exact economics, and
the scarce-vacancy smoke test's `n_agents=18,000` threshold is specific to it too. Swapping in
3.17% now would silently invalidate both without re-validation. The real rate is computed, checked,
and recorded here; adopting it into the configs is deliberately left for Week 4, alongside the
rest of the calibration work, where those artefacts get re-checked as a matter of course rather
than as an unplanned side effect of this notebook."

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings(
    "ignore", category=UnicodeWarning
)  # QLFS string fields fall back to latin-1; expected, not a bug

# Edit this to wherever you extracted the DataFirst downloads -- see data/README.md.
DATA_ROOT = Path(r"C:\Users\aakas\Documents\Projects\Thesis\Data\extracted")

QUARTERS = {
    "2025-Q2": DATA_ROOT / "qlfs-2025-q2-v1" / "qlfs-2025-q2-v1.dta",
    "2025-Q3": DATA_ROOT / "qlfs-2025-03" / "QLFS202503.dta",
    "2025-Q4": DATA_ROOT / "qlfs-2025-04" / "qlfs-2025-q4-v1.dta",
    "2026-Q1": DATA_ROOT / "qlfs-2026-q1-v1" / "qlfs-2026-q1-v1.dta",
}
LONG_TERM_CATEGORIES = [
    "Long-term unemployment (1 year and longer)",
    "Short-term unemployment (less than 1 year)",
]


def weighted_share(df: pd.DataFrame, flag_col: str, flag_value, weight_col: str = "Weight"):
    w = df[weight_col].to_numpy()
    is_flag = (df[flag_col] == flag_value).to_numpy().astype(float)
    p_hat = np.average(is_flag, weights=w)
    n = len(df)
    deff = (w**2).sum() * n / (w.sum() ** 2)
    se = np.sqrt(p_hat * (1 - p_hat) / (n / deff))
    return p_hat, se, n


results = {}
for quarter, path in QUARTERS.items():
    df = pd.read_stata(path, columns=["Long_term_unempl", "Weight"], convert_categoricals=True)
    sub = df[df["Long_term_unempl"].isin(LONG_TERM_CATEGORIES)]
    p_hat, se, n = weighted_share(sub, "Long_term_unempl", LONG_TERM_CATEGORIES[0])
    results[quarter] = {"n": n, "long_term_share": p_hat, "se": se}
    print(f"{quarter}: n={n:,}  long_term_share={p_hat:.4f}  se={se:.4f}")

results_df = pd.DataFrame(results).T

In [ ]:
headline_quarter = "2026-Q1"
headline = results_df.loc[headline_quarter]

moments_path = Path("../../data/moments.csv")
moments = pd.read_csv(moments_path)
moments["period"] = moments["period"].astype("object")
moments["source"] = moments["source"].astype("object")
row = moments["key"] == "long_term_share"
moments.loc[row, "value"] = round(float(headline["long_term_share"]), 4)
moments.loc[row, "standard_error"] = round(float(headline["se"]), 4)
moments.loc[row, "period"] = headline_quarter
moments.loc[row, "source"] = (
    "Statistics South Africa. Quarterly Labour Force Survey 2026: Q1 [dataset]. "
    "Cape Town: DataFirst [distributor]. QLFS Long_term_unempl variable, among the "
    "currently unemployed, person-weighted."
)
moments.loc[row, "provisional"] = False
moments.to_csv(moments_path, index=False)
moments[row]

## Part B: building the QLFS panel

Loads the same four quarters, this time keeping `UQNO` and `PERSONNO` to build a within-person
identifier and link consecutive quarters.

In [ ]:
panel_frames = {}
for quarter, path in QUARTERS.items():
    df = pd.read_stata(
        path, columns=["UQNO", "PERSONNO", "Status", "Weight"], convert_categoricals=True
    )
    df["pid"] = df["UQNO"].astype(str) + "_" + df["PERSONNO"].astype(str)
    panel_frames[quarter] = df.set_index("pid")

quarter_labels = list(QUARTERS.keys())
for t0, t1 in zip(quarter_labels[:-1], quarter_labels[1:], strict=True):
    overlap = panel_frames[t0].index.intersection(panel_frames[t1].index)
    print(f"{t0} -> {t1}: {len(overlap):,} matched of {len(panel_frames[t0]):,} in {t0}")

In [ ]:
pair_rates = []
all_sep_flags = []
all_weights = []
for t0, t1 in zip(quarter_labels[:-1], quarter_labels[1:], strict=True):
    a, b = panel_frames[t0], panel_frames[t1]
    common = a.index.intersection(b.index)
    a_c, b_c = a.loc[common], b.loc[common]

    employed_t0 = (a_c["Status"] == "Employed").to_numpy()
    separated = employed_t0 & (b_c["Status"] != "Employed").to_numpy()
    w = a_c["Weight"].to_numpy()

    rate = np.average(separated[employed_t0].astype(float), weights=w[employed_t0])
    pair_rates.append(rate)
    all_sep_flags.append(separated[employed_t0])
    all_weights.append(w[employed_t0])
    print(f"{t0} -> {t1}: n_employed={employed_t0.sum():,}  quarterly_separation_rate={rate:.4f}")

sep_flags = np.concatenate(all_sep_flags).astype(float)
sep_weights = np.concatenate(all_weights)
pooled_quarterly_rate = np.average(sep_flags, weights=sep_weights)
print(f"\npooled quarterly separation rate: {pooled_quarterly_rate:.4f}")

In [ ]:
rng = np.random.default_rng(42)
n_boot = 2000
sep_p = sep_weights / sep_weights.sum()
boot_quarterly = np.empty(n_boot)
for i in range(n_boot):
    idx = rng.choice(len(sep_flags), size=len(sep_flags), replace=True, p=sep_p)
    boot_quarterly[i] = sep_flags[idx].mean()

# Same quarterly-to-monthly compounding convention as rho_A (DECISIONS.md's AR(1) note):
# (1 - monthly)**3 = (1 - quarterly).
monthly_rate = 1 - (1 - pooled_quarterly_rate) ** (1 / 3)
boot_monthly = 1 - (1 - boot_quarterly) ** (1 / 3)
quarterly_se = boot_quarterly.std(ddof=1)
monthly_se = boot_monthly.std(ddof=1)
ci_lo, ci_hi = np.percentile(boot_monthly, [2.5, 97.5])
fallback_ratio = monthly_rate / 0.0048

print(f"quarterly separation rate: {pooled_quarterly_rate:.4f} (bootstrap SE {quarterly_se:.4f})")
print(f"monthly separation rate:   {monthly_rate:.4f} (bootstrap SE {monthly_se:.4f})")
print(f"95% bootstrap CI (monthly): [{ci_lo:.4f}, {ci_hi:.4f}]")
print(f"\nMiyamoto (2011)'s Japanese fallback: 0.0048 -- this rate is {fallback_ratio:.1f}x higher")

## Result

**long_term_share**: written to `moments.csv` above, most recent quarter as headline (same
convention as notebook 01).

**Separation rate**: 3.17% monthly (95% CI 3.06-3.29%), computed and validated here, but **not
yet written into any config file** -- see the reasoning in the first markdown cell.
`configs/baseline.yaml` still reads `separation_rate: 0.0048 # Miyamoto (2011)'s fallback`;
that line should change to this notebook's value as part of Week 4's calibration setup, at
which point `trace_demo.yaml`'s seed/agent scan and the scarce-vacancy smoke test's population
threshold both need a quick re-check under the new economics.